# llmpic V0.3.0 — Natural Language → Production Charts

> Describe what you want. Get a chart. **12 chart types.** PNG / SVG / PDF. Jupyter inline display. Async batch. Iterative editing. Geographic maps. Sandbox safety.

[GitHub](https://github.com/ADW-19/llmpic) · [Docs](https://ADW-19.github.io/llmpic/) · [PyPI](https://pypi.org/project/llmpic/)

## 1. Initialize SDK

Pick any OpenAI-compatible endpoint: OpenAI, DeepSeek, Azure, GLM, Ollama, vLLM...

In [ ]:
import os
from llmpic import llmPIC

lp = llmPIC(
    api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
    base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
    model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    max_tokens=4096,
)
print(f"SDK ready — model: {lp.model}")

## 2. Quick Start — One Line, One Chart

Without any data, the LLM auto-generates realistic demo data with numpy.

In [ ]:
r = lp.plot("Monthly sales trend over the past 12 months").render()

if r.success:
    r.show()  # Renders inline right below the cell
    print(f"Size: {r.size_kb:.0f}KB | Tokens: in={r.token_usage['input']} out={r.token_usage['output']}")
else:
    print(f"Failed: {r.error_message}")

## 3. Data Input — 6 Ways to Feed Data

llmpic accepts virtually any data format. The LLM receives a serialized summary (column names, types, first N rows) and writes code using your exact column names.

### 3.1 No Data — Auto-Generated

In [ ]:
lp.plot("CPU usage over 30 days, two alternating peaks per day").render().show()

### 3.2 Inline Data — Numbers right in the query

In [ ]:
lp.bar("Q1 Budget: R&D=200K, Marketing=150K, Sales=180K, HR=100K").render().show()

### 3.3 pandas DataFrame — Automatic column recognition

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "Month": pd.date_range("2025-01-01", periods=12, freq="MS").strftime("%b"),
    "Revenue": np.random.randint(80, 200, 12),
    "Cost": np.random.randint(50, 120, 12),
})
df["Profit"] = df["Revenue"] - df["Cost"]

lp.bar("Revenue vs Cost vs Profit by month, grouped bars").data(df).render().show()

### 3.4 NumPy Array

In [ ]:
data = np.random.randn(1000)
lp.hist("Score distribution, 30 bins, KDE overlay").data(data).render().show()

### 3.5 Python Dict / List

In [ ]:
lp.pie("Market share distribution").data({
    "Product A": 35, "Product B": 28, "Product C": 18, "Product D": 12, "Others": 7
}).render().show()

lp.plot("Weekly temperature").data([22, 24, 19, 26, 28, 25, 23]).render().show()

### 3.6 CSV / Excel — via pandas

In [ ]:
# df = pd.read_csv("your_data.csv")
# lp.scatter("Age vs income correlation").data(df).render().show()

## 4. Style Customization

6 built-in color schemes + 10 style parameters. Chain `.style()` before `.render()`.

### 4.1 Preset Color Schemes

In [ ]:
from IPython.display import display, Markdown

for scheme in ["blues", "warm", "cool", "pastel", "dark", "grayscale"]:
    r = lp.bar(f"Q4 Revenue by Product: Cloud=450, AI=320, Security=280").style({"color_scheme": scheme}).render()
    if r.success:
        display(Markdown(f"**{scheme}**"))
        r.show()

### 4.2 All Style Parameters

In [ ]:
lp.plot("Revenue trend over 12 months").data(df).style({
    "figsize": [14, 7],         # Width, height in inches
    "dpi": 200,                  # Output resolution
    "color_scheme": "blues",     # blues | warm | cool | pastel | dark | grayscale
    "title_fontsize": 18,        # Chart title font size
    "label_fontsize": 14,        # Axis label font size
    "tick_fontsize": 12,         # Tick mark font size
    "grid": True,                # Show background grid
    "grid_alpha": 0.3,           # Grid line transparency (0-1)
    "tight_layout": True,        # Auto-adjust layout
    "facecolor": "#FAFAFA",      # Figure background color
}).render().show()

## 5. All 12 Chart Types

In [ ]:
from IPython.display import display, Markdown

charts = [
    ("1. plot — Line", lp.plot("2024 monthly sales trend, 12 months")),
    ("2. scatter — Scatter", lp.scatter("Random scatter: 50 points, Age vs Income, trend line")),
    ("3. bar — Bar", lp.bar("Department Budget: R&D=200K, Marketing=150K, Sales=180K, HR=100K")),
    ("4. pie — Pie", lp.pie("Market share: A=40%, B=25%, C=20%, Others=15%")),
    ("5. hist — Histogram", lp.hist("Normal distribution N(0,1), 1000 samples, KDE overlay")),
    ("6. heatmap — Heatmap", lp.heatmap("6x6 correlation matrix, annotated, coolwarm")),
    ("7. boxplot — Boxplot", lp.boxplot("Test scores: Group A, B, C, D (4 groups, 30 each)")),
    ("8. area — Area", lp.area("Revenue composition by 3 product lines 2020-2024, stacked")),
    ("9. radar — Radar", lp.radar("Product: Performance=4, Usability=3, Reliability=5, Price=2, Support=4")),
    ("10. subplots — Dashboard", lp.subplots("2x2: sales line, region bar, customer scatter, growth hist")),
    ("11. custom — Auto-detect", lp.custom("Analyze user retention trend with multiple factors")),
    ("12. map — Geographic Map (v0.3.0)", lp.map("World major cities population, Blues colormap")),
]

for title, builder in charts:
    r = builder.render()
    display(Markdown(f"### {title}"))
    if r.success:
        r.show()
        print(f"   {r.size_kb:.0f}KB | tokens in={r.token_usage['input']} out={r.token_usage['output']}")
    else:
        print(f"   FAIL: {r.error_message[:120]}")

## 6. ChartResult — Inspect & Export

### 6.1 Basic Properties

In [ ]:
r = lp.plot("sin(x) from 0 to 2π, smooth curve").render()

print(f"Success:      {r.success}")
print(f"Size:         {r.size_kb:.1f} KB")
print(f"Format:       {r._format}")
print(f"Tokens in:    {r.token_usage['input']}")
print(f"Tokens out:   {r.token_usage['output']}")
print(f"Code length:  {len(r.code)} chars")

print(f"\n--- Generated Code (first 500 chars) ---\n{r.code[:500]}...")

### 6.2 Save to File — PNG / SVG / PDF

In [ ]:
r = lp.plot("CPU usage over 24 hours, peaks at 9am and 3pm").render()

path1 = r.save("cpu_usage.png")   # PNG
path2 = r.save("cpu_usage.svg")   # SVG vector
path3 = r.save("cpu_usage.pdf")   # PDF
path4 = r.save()                   # auto: ~/llmpic_charts/chart_{timestamp}.png

print(f"Saved:\n  {path1}\n  {path2}\n  {path3}\n  {path4}")

### 6.3 Base64 Encoded — Web Embedding

In [ ]:
r = lp.pie("Market: A=40%, B=30%, C=20%, D=10%").render()

png_b64 = r.base64()      # data:image/png;base64,...
svg_b64 = r.base64_svg()  # data:image/svg+xml;base64,...

print(f"PNG base64: {len(png_b64):,} chars")
print(f"SVG base64: {len(svg_b64):,} chars")

### 6.4 Lazy Format Access — SVG / PDF from same result

In [ ]:
r = lp.plot("Monthly sales trend").render()  # Default: PNG

svg_bytes = r.svg_bytes   # Re-renders as SVG on first access, cached
pdf_bytes = r.pdf_bytes   # Re-renders as PDF
svg_str   = r.svg         # SVG as Python string

print(f"PNG:  {r.size_kb:.0f} KB")
print(f"SVG:  {len(svg_bytes) / 1024:.0f} KB")
print(f"PDF:  {len(pdf_bytes) / 1024:.0f} KB")

r.show()

## 7. Iterative Editing — Refine with Natural Language

`.edit()` sends the current code + your edit request to the LLM. Returns a **new** ChartResult — originals are never mutated.

In [ ]:
from IPython.display import display, Markdown

v1 = lp.plot("Quarterly sales: Q1=100, Q2=150, Q3=120, Q4=180").render()
display(Markdown("### v1 — Initial line chart"))
v1.show()

v2 = v1.edit("Change to bar chart, use blue tones")
display(Markdown("### v2 — Bar chart + blue"))
v2.show()

v3 = v2.edit("Title '2025 Annual Sales Report', increase title size to 18, add grid lines")
display(Markdown("### v3 — Refined title + grid"))
v3.show()

v4 = v3.edit("Switch to warm color scheme, add y-axis label 'Revenue (K USD)'")
display(Markdown("### v4 — Final"))
v4.show()

v4.save("final_report.png")
print("Saved: final_report.png")

## 8. Safety Configuration

Two safety modes. The sandbox already blocks all execution paths — **fast** mode is sufficient for production.

In [ ]:
# Fast mode (default) — 32 precompiled regex patterns, ~0ms overhead
lp_fast = llmPIC(
    api_key="sk-...", base_url="https://api...",
    safety_level="fast",
)

# Full mode — regex + LLM semantic review, adds ~1-2s per chart
lp_full = llmPIC(
    api_key="sk-...", base_url="https://api...",
    safety_level="full",
)

print("Both modes ready. fast=regex only, full=regex+LLM review.")

## 9. Async Batch — Concurrent Generation

All charts run in parallel. Total time ≈ slowest single chart.

In [ ]:
from llmpic import AsyncllmPIC
from IPython.display import display, Markdown
import time

async def run_batch():
    lp_async = AsyncllmPIC(
        api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
        base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
        model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    )

    t0 = time.time()
    results = await lp_async.batch([
        ("plot",     "National 12-month sales trend"),
        ("bar",      "Sales comparison by region"),
        ("pie",      "Market share distribution"),
        ("scatter",  "Customer age vs annual spend"),
        ("heatmap",  "5x5 correlation matrix"),
    ])

    for i, r in enumerate(results):
        if r.success:
            display(Markdown(f"### batch[{i}] — {r.size_kb:.0f}KB"))
            r.show()
        else:
            print(f"[{i}] FAILED: {r.error_message[:100]}")

    print(f"\n5 charts concurrently, total time: {time.time()-t0:.1f}s")

await run_batch()

## 10. Advanced — Custom Chart Type & Code Access

Let the LLM decide the best chart type. Access the generated matplotlib code for manual tuning.

In [ ]:
r = lp.custom("Analyze user retention rate changes and contributing factors").render()
r.show()

print("=== Generated Code ===")
print(r.code)

print(f"\n=== Token Usage ===")
print(f"Input: {r.token_usage['input']}, Output: {r.token_usage['output']}")

## 11. Format Conversion — PNG → SVG → PDF on the Same Result

One LLM call. Three formats. Lazy, cached re-rendering from stored code.

In [ ]:
r = lp.bar("Sales by region: North=320, South=280, East=260, West=200").render()

print(f"PNG: {r.size_kb:.0f}KB ({len(r.image_bytes):,} bytes)")
print(f"SVG: {len(r.svg_bytes) / 1024:.0f}KB")
print(f"PDF: {len(r.pdf_bytes) / 1024:.0f}KB")
print(f"\nGenerated {len(r.code)} chars of matplotlib code, rendered 3 formats")

---

**That's it!** You've covered all llmpic V0.3.0 APIs:

| API | Section |
|-----|---------|
| `llmPIC()`, `AsyncllmPIC()` | 1, 9 |
| `.plot()` `.scatter()` `.bar()` `.pie()` `.hist()` `.heatmap()` `.boxplot()` `.area()` `.radar()` `.map()` `.subplots()` `.custom()` | 2, 5 |
| `.data()` | 3 |
| `.style()` | 4 |
| `.render()` `.save()` | 2, 6 |
| `.show()` `.base64()` `.base64_svg()` | 2, 6 |
| `.edit()` | 7 |
| `safety_level` | 8 |
| `.batch()` | 9 |
| `.code` `.token_usage` `.size_kb` `.svg_bytes` `.pdf_bytes` `.svg` | 6, 10 |

⭐ [GitHub](https://github.com/ADW-19/llmpic) · [Docs](https://ADW-19.github.io/llmpic/) · [PyPI](https://pypi.org/project/llmpic/)